# Scalable Network Intrusion Detection Using Apache Spark
### Interactive Analysis & Model Evaluation Notebook
This notebook demonstrates loading the preprocessed **CICIDS2017** dataset, evaluating the trained **PySpark MLlib** models, and running real-time network flow inference.

In [ ]:
import os
import sys
import json
import pandas as pd

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.spark.spark_session import get_spark_session, stop_spark_session
from src.models.predictor import load_inference_artifacts, predict_single_flow, predict_batch_flows

# Initialize Spark Session
spark = get_spark_session('NotebookAnalysis')
print(f"Spark Session Active: v{spark.version}")

## 1. Load Preprocessed Test Partition

In [ ]:
test_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test.parquet')
test_df = spark.read.parquet(test_path)
print(f"Total Test Records: {test_df.count():,}")
test_df.select('destination_port', 'flow_duration', 'total_fwd_packets', 'label').show(5)

## 2. Load Model Benchmark Results

In [ ]:
report_path = os.path.join(PROJECT_ROOT, 'results', 'model_comparison.json')
with open(report_path) as f:
    comparison = json.load(f)

results_table = []
for m in comparison:
    results_table.append({
        'Model': m['model_name'],
        'Accuracy (%)': f"{m['accuracy']*100:.2f}%",
        'Precision (%)': f"{m['precision']*100:.2f}%",
        'Recall (%)': f"{m['recall']*100:.2f}%",
        'F1-Score (%)': f"{m['f1_score']*100:.2f}%",
        'ROC-AUC': m['roc_auc']
    })
pd.DataFrame(results_table)

## 3. Interactive Single Flow Prediction

In [ ]:
presets_path = os.path.join(PROJECT_ROOT, 'results', 'sample_presets.json')
with open(presets_path) as f:
    presets = json.load(f)

# Test Benign Flow
benign_result = predict_single_flow(presets['BENIGN'], model_name='Random Forest')
print("Benign Preset Prediction:", benign_result)

# Test PortScan Attack Flow
attack_result = predict_single_flow(presets['PORTSCAN'], model_name='Random Forest')
print("PortScan Preset Prediction:", attack_result)

## 4. Batch Classification on Demo Dataset

In [ ]:
demo_csv = os.path.join(PROJECT_ROOT, 'data', 'processed', 'demo_batch_sample.csv')
demo_df = pd.read_csv(demo_csv).head(100)

predictions, summary = predict_batch_flows(demo_df, model_name='Random Forest')
print("Batch Summary:", summary)
predictions[['Predicted_Class', 'Threat_Score (%)', 'Confidence (%)', 'destination_port', 'flow_duration']].head(10)